# Lesson 2 : LangGraph Components

In [ ]:
from dotenv import load_dotenv
_ = load_dotenv()

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools import DuckDuckGoSearchRun

In [2]:
#tool = TavilySearchResults(max_results=4) #increased number of results
tool = DuckDuckGoSearchRun()
results = tool.invoke("Obama's first name?")
print(results)
print(type(tool))
print(tool.name)

In February 1981, Obama made his first public speech, calling for Occidental to participate in the disinvestment from South Africa in response to ... ... searching our database we found 1 possible solution for the: ... The solution we have for President Obama ' s first name has a total of 6 letters. According to the BBC , Odinga claims Obama as a first cousin.) We should eschew guilt by association, but the echoes resound. For many, the name was exotic, signifying a kind of transformation of the customary, the core of Obama ’ s schtick which brought us to the brink of ... Oh did we mention that Dorne Ayers Wif got Michille Obamas Wife her first Job at a Law Firm in Chicago and the whole Bunch lives in the same ...
<class 'langchain_community.tools.ddg_search.tool.DuckDuckGoSearchRun'>
duckduckgo_search


> If you are not familiar with python typing annotation, you can refer to the [python documents](https://docs.python.org/3/library/typing.html).

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

> Note: in `take_action` below, some logic was added to cover the case that the LLM returned a non-existent tool name. Even with function calling, LLMs can still occasionally hallucinate. Note that all that is done is instructing the LLM to try again! An advantage of an agentic organization.

In [4]:
from langchain_core.tools import Tool
    
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
    
        
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0
        
        
    
    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [7]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
base_url = "http://127.0.0.1:8888/v1" 
model_name ="google/gemma-3-4b"

#model = ChatOpenAI(model="gpt-3.5-turbo")  #reduce inference cost
model = ChatOpenAI(model=model_name, temperature=0, max_tokens=None, timeout=None, max_retries=2,api_key="llm-studio", base_url=base_url)
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
print(ai_msg)
abot = Agent(model, [tool], system=prompt)

content='Je suis passionné(e) par la programmation. \n\n**(Note:** I added "(e)" because if you\'re female, you would say "passionnée".)' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 30, 'total_tokens': 68, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-zu7tjziezfd4vtc8sca5gs', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--6a35a386-3988-4bec-9112-6db58d71c2b8-0' usage_metadata={'input_tokens': 30, 'output_tokens': 38, 'total_tokens': 68, 'input_token_details': {}, 'output_token_details': {}}


In [ ]:
from IPython.display import Image

Image(abot.graph.get_graph().draw_png())

In [6]:
messages = [HumanMessage(content="What is the weather in sf?")]
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in sf'}, 'id': '435551254', 'type': 'tool_call'}
Back to the model!


In [7]:
result

{'messages': [HumanMessage(content='What is the weather in sf?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '435551254', 'function': {'arguments': '{"query":"weather in sf"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 498, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-4yl41nli0ks40h8vkmqh4a', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--bfee0943-458b-4314-a758-b85a96de2f37-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in sf'}, 'id': '435551254', 'type': 'tool_call'}], usage_metadata={'input_tokens': 498, 'output_tokens': 35, 'total_tokens': 533, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(conten

In [8]:
result['messages'][-1].content

'The current weather in San Francisco, CA is 64°F (18°C) with partly cloudy skies. The forecast for the next three days includes sunny conditions with highs ranging from 72°F to 75°F (22°C to 24°C).'

In [9]:
messages = [HumanMessage(content="What is the weather in SF and LA?")]
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in San Francisco'}, 'id': '353906334', 'type': 'tool_call'}
Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in Los Angeles'}, 'id': '249234454', 'type': 'tool_call'}
Back to the model!


In [10]:
result['messages'][-1].content

"Okay, here's the weather information for San Francisco and Los Angeles:\n\nSan Francisco is expecting partly cloudy skies early that will become partly cloudy in the afternoon.\n\nLos Angeles is currently showing data from various sources including METAR reports and NOAA Weather data. It appears there may be some issues with weather radar functionality within XPlane 12, but this doesn't affect the current weather conditions."

In [15]:
# Note, the query was modified to produce more consistent results. 
# Results may vary per run and over time as search information and models change.

query = "Who won the super bowl in 2024? In what state is the winning team headquarters located? \
What is the GDP of that state? Answer both question." 
messages = [HumanMessage(content=query)]

#model = ChatOpenAI(model="gpt-4o")  # requires more advanced model
model = ChatOpenAI(model=model_name, temperature=0, max_tokens=None, timeout=None, max_retries=2,api_key="llm-studio", base_url=base_url)
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'duckduckgo_search', 'args': {'query': 'super bowl 2024 winner'}, 'id': '987863844', 'type': 'tool_call'}
Back to the model!
Calling: {'name': 'duckduckgo_search', 'args': {'query': 'GDP of Missouri'}, 'id': '716463493', 'type': 'tool_call'}
Back to the model!


In [16]:
print(result['messages'][-1].content)

The GDP of Missouri is $356.65 billion.


In [16]:
query = "What is 1+5*30 in python" 
messages = [HumanMessage(content=query)]

#model = ChatOpenAI(model="gpt-4o")  # requires more advanced model
model = ChatOpenAI(model=model_name, temperature=0, max_tokens=None, timeout=None, max_retries=2,api_key="llm-studio", base_url=base_url)
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})
print(result['messages'][-1].content)

Calling: {'name': 'duckduckgo_search', 'args': {'query': '1+5*30 python'}, 'id': '291440974', 'type': 'tool_call'}
Back to the model!
```python
1 + 5 * 30
```
